# 05. V1 Retrieval 패턴 & Hybrid RAG

> 리트리버 자체에도 디자인이 필요해요. 2-Step / Agentic / Hybrid 세 가지 retrieval 패턴과 쿼리 강화 · MMR 같은 검색 품질 트릭을 묶어 정리해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **2-Step RAG** 패턴으로 검색과 생성을 분리하여 LangGraph 노드로 구현할 수 있어요
2. **Agentic RAG** 패턴으로 `create_agent`와 `@tool` 데코레이터를 활용한 도구 기반 검색 에이전트를 만들 수 있어요
3. **Hybrid RAG** 패턴으로 쿼리 강화(Query Enhancement)와 검증 루프를 통해 다중 전략 검색을 구현할 수 있어요
4. **FAISS** 벡터스토어와 `as_retriever()`로 다양한 검색 파라미터를 조정할 수 있어요
5. 메타데이터 필터링으로 소스, 페이지 범위별 검색 범위를 제한할 수 있어요

## 사전 지식

- 이전 노트북 `04-Adaptive-RAG.ipynb`에서 배운 라우팅 + 평가 기반 RAG 패턴
- `FAISS.from_documents()`, `OpenAIEmbeddings`, `RecursiveCharacterTextSplitter` 기본 사용법
- LangGraph `StateGraph`, 조건부 엣지, `create_agent` 기본 개념

## RAG 패턴 비교

이 노트북에서는 세 가지 Retrieval 패턴을 비교하며 배워요. 각각의 장단점과 적합한 사용 시나리오가 달라요.

> 🔑 **핵심 개념**: 이전 노트북들(01-04)에서는 "검색 결과를 어떻게 검증하고 보강할까?"에 집중했어요. 이 노트북에서는 한 걸음 더 나아가 "검색 자체를 어떻게 더 잘할까?"를 다뤄요. 같은 FAISS 인덱스라도 **어떻게 질문하느냐**에 따라 검색 결과가 완전히 달라져요.

### 세 가지 패턴 비교

| 패턴 | 검색 전략 | 비유 | 적합한 시나리오 |
|------|----------|------|----------------|
| **2-Step RAG** | 단일 벡터 검색 | 도서관에서 제목 하나로 검색 | 문서가 잘 정리된 경우 |
| **Agentic RAG** | 에이전트가 도구 결정 | 사서에게 질문하면 알아서 찾아줌 | 복잡한 질문, 다중 소스 |
| **Hybrid RAG** | 쿼리 강화 + 다중 검색 | 같은 질문을 여러 방식으로 검색 | 정확도가 중요한 경우 |

> 💡 **핵심 정리**: 세 패턴 모두 같은 FAISS 인덱스를 사용하지만, **어떻게 검색하고 검증하는지**가 달라요. 2-Step은 "한 번 묻고 끝", Agentic은 "필요할 때만 묻기", Hybrid는 "같은 질문을 여러 각도로 묻기"예요.

### 전체 아키텍처

```mermaid
flowchart LR
    subgraph 공통 인프라
        D[(FAISS<br/>벡터스토어)] 
        E[OpenAI<br/>Embeddings]
    end

    subgraph 패턴1[2-Step RAG]
        A1[질문] --> B1[retrieve<br/>검색]
        B1 --> C1[generate<br/>생성]
    end

    subgraph 패턴2[Agentic RAG]
        A2[질문] --> B2[agent<br/>도구 결정]
        B2 -- 검색필요 --> C2[tool<br/>실행]
        C2 --> B2
        B2 -- 완료 --> D2[최종답변]
    end

    subgraph 패턴3[Hybrid RAG]
        A3[질문] --> B3[query<br/>enhancement]
        B3 --> C3[multi<br/>retrieve]
        C3 --> E3{검증}
        E3 -- 불충분 --> B3
        E3 -- 충분 --> F3[generate]
    end

    D --> B1
    D --> C2
    D --> C3

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A1,A2,A3 input
    class B1,B2,C2,B3,C3,E3 process
    class C1,D2,F3 output
    class D,E storage
```

## 1. 환경 설정

In [ ]:
from dotenv import load_dotenv

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택 사항)
# ---------------------------------------------------
# LangSmith를 사용하면 그래프 실행을 시각적으로 추적할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-RAG-Retrieval"

## 2. 공통 인프라: FAISS 벡터스토어 구성

세 가지 패턴 모두 동일한 FAISS 인덱스를 사용해요. PDF를 로드하고 청크로 분할한 뒤 임베딩하여 검색 가능한 벡터스토어를 만들어요.

> 💡 **실무 팁**: `chunk_size`와 `chunk_overlap`은 검색 품질에 큰 영향을 미쳐요. 너무 작으면 문맥이 잘리고, 너무 크면 관련 없는 내용이 섞여요. 일반적으로 500~1000자 크기에 10~20% 오버랩이 좋은 출발점이에요.

**실습 문서**: 소프트웨어정책연구소(SPRi) AI Brief 2023년 12월호

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import os

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 retriever (유사도 기반, 상위 4개)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 3. 패턴 1: 2-Step RAG

가장 단순하고 직관적인 RAG 패턴이에요. **검색(Retrieve)** 노드와 **생성(Generate)** 노드가 순차적으로 실행되는 선형 파이프라인이에요.

### 2-Step RAG의 흐름

```mermaid
flowchart LR
    A([START]) --> B[retrieve<br>벡터 검색]
    B --> C[generate<br>답변 생성]
    C --> D([END])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef terminal fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A,D terminal
    class B process
    class C output
```

> 💡 **핵심 정리**: 2-Step RAG의 단순함이 오히려 장점이에요. 디버깅이 쉽고, 각 단계를 독립적으로 테스트할 수 있어요. 기본적인 Q&A 챗봇에는 이 패턴으로 충분한 경우가 많아요.

> ⚠️ **자주 하는 실수**: `question` 키를 `State`에 포함하지 않으면 노드 간 질문 전달이 안 돼요. `messages` 리스트로만 관리하다가 `retrieve` 노드에서 어떤 질문인지 알 수 없게 되는 경우가 자주 발생해요.

In [ ]:
from typing import Annotated, List
from typing_extensions import TypedDict
from langchain_core.documents import Document

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph.graph import END, StateGraph, START

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → retrieve → generate → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import uuid
from langchain_core.runnables import RunnableConfig

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트 질문
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 4. 패턴 2: Agentic RAG

**Agentic RAG**는 `create_agent`로 생성된 에이전트가 검색 도구 사용 여부를 스스로 판단해요. 에이전트는 질문에 따라 도구를 호출하거나 LLM의 내장 지식으로 직접 답변할 수 있어요.

### Agentic RAG의 특징

| 항목 | 설명 |
|------|------|
| **검색 주체** | 에이전트가 자율적으로 결정 |
| **도구 정의** | `@tool` 데코레이터로 Python 함수를 도구로 변환 |
| **도구 선택** | LLM이 질문과 도구 설명을 비교해 선택 |
| **멀티턴** | 한 번에 여러 도구를 순차적으로 호출 가능 |

> 🔑 **핵심 개념**: `@tool` 데코레이터의 **docstring**이 도구 선택의 핵심이에요. LLM이 docstring을 읽고 이 도구를 언제 사용할지 결정해요. 영어로 명확하게 작성하는 것이 중요해요.

### Agentic RAG 흐름

```mermaid
flowchart TD
    A([START]) --> B[agent<br>create_agent]
    B -- 도구 호출 결정 --> C[tools<br>@tool 실행]
    C -- 결과 반환 --> B
    B -- 완료 --> D([END])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef terminal fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A,D terminal
    class B,C process
```

> 💡 **실무 팁**: `create_agent`를 사용하면 LangGraph의 내부 루프 로직을 직접 구현하지 않아도 돼요. 도구 목록과 시스템 프롬프트만 정의하면 자동으로 ReAct 패턴(추론 → 행동 → 관찰)을 구현해줘요.

In [ ]:
from langchain.tools import tool

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트가 사용할 도구 목록
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.agents import create_agent

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 시스템 프롬프트 (역할과 행동 지침)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → tools → agent → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain_core.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 5. 패턴 3: Hybrid RAG

**Hybrid RAG**는 **쿼리 강화(Query Enhancement)**와 **검증 루프(Validation Loop)**를 결합한 고급 패턴이에요. 단일 쿼리로 검색하는 대신, 원본 쿼리를 다양한 방식으로 확장하여 더 많은 관련 문서를 찾아요.

> 🔑 **핵심 개념**: Hybrid RAG는 **형사 수사**와 비슷해요. 단서 하나(원본 쿼리)로 수사하는 것보다, 같은 사건을 여러 각도에서 조사(강화된 쿼리들)하면 더 많은 증거(관련 문서)를 찾을 수 있어요. "삼성전자 AI"라는 하나의 질문을 "삼성 가우스", "삼성 생성형 AI 모델", "Samsung AI development" 등으로 다각화하면 놓치는 문서가 줄어들어요.

### Hybrid RAG의 핵심 아이디어

1. **쿼리 강화**: 원본 질문을 3-5개의 다양한 관점으로 재작성해요
2. **다중 검색**: 각 강화된 쿼리로 개별 검색 수행 (재현율 향상)
3. **중복 제거**: 동일 청크가 여러 번 검색되면 한 번만 사용해요
4. **검증**: 충분한 문서가 모였는지 확인, 아니면 재시도

### 쿼리 강화가 왜 효과적인가요?

| 원본 쿼리 | 강화된 쿼리 예시 | 새로 찾을 수 있는 정보 |
|----------|----------------|---------------------|
| "삼성전자 AI" | "삼성 가우스 생성형 AI 특징" | 삼성 가우스의 기술적 특징 |
| | "Samsung Electronics AI investment 2023" | 삼성의 AI 투자 관련 영문 정보 |
| | "삼성전자 AI 모델 출시 일정" | 모델 출시 타임라인 |

> 💡 **실무 팁**: 단일 쿼리로 검색할 때는 특정 단어가 포함된 문서만 찾을 수 있어요. 쿼리를 다각화하면 같은 주제를 다른 표현으로 담은 문서까지 포함할 수 있어요. 이것이 Hybrid RAG의 핵심이에요.

### Hybrid RAG 흐름

```mermaid
flowchart TD
    A([START]) --> B[enhance_query<br/>쿼리 강화]
    B --> C[multi_retrieve<br/>다중 검색 + 중복제거]
    C --> D{validate_results<br/>검증}
    D -- 불충분 --> B
    D -- 충분 --> E[generate<br/>답변 생성]
    E --> F([END])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef decision fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef output fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef terminal fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A,F terminal
    class B,C process
    class D decision
    class E output
```

> ⚠️ **자주 하는 실수**: 검증 루프에서 `retry_count`를 관리하지 않으면 무한 루프가 발생할 수 있어요. `max_retries` 제한을 반드시 설정해요.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트: 쿼리 강화 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from typing import Literal

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → enhance_query → multi_retrieve → (enhance_query | generate) → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 6. 메타데이터 필터링

**메타데이터 필터링**은 검색 범위를 특정 소스, 날짜, 페이지 번호 등으로 제한하는 강력한 기법이에요. 특히 다중 문서를 인덱싱할 때 특정 문서에서만 검색하고 싶을 때 유용해요.

> 💡 **핵심 정리**: 메타데이터 필터링은 프로덕션 RAG 시스템에서 매우 중요한 기법이에요. 여러 고객사의 문서를 한 인덱스에 저장하고, 각 고객의 질문은 해당 고객의 문서에서만 검색하도록 격리할 수 있어요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트: 정책/법제 섹션(1-6페이지)에서만 검색
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 7. 세 패턴 비교 실험

> 💡 **핵심 정리**: 동일한 질문으로 세 패턴을 비교하면서 각각의 특성을 직접 확인해볼게요. 속도, 정확도, 컨텍스트 풍부함에서 차이가 나는 것을 확인해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 세 패턴 비교 실험
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 8. MMR 검색: 다양성 보장 검색

**MMR(Maximal Marginal Relevance)**는 관련성과 다양성을 동시에 고려하는 검색 방법이에요. 유사한 내용의 문서가 반복 선택되는 것을 방지해요.

> 🔑 **핵심 개념**: 일반 유사도 검색은 "가장 비슷한 문서 4개"를 찾아요. 하지만 같은 문단에서 잘린 청크 3개가 선택되면 **사실상 같은 내용을 3번 보는 꼴**이에요. MMR은 **뉴스 큐레이션**처럼 동작해요 — 같은 사건의 기사 10개가 있을 때, 서로 다른 관점의 기사 4개를 골라주는 거예요.

### 유사도 검색 vs MMR 비교

| 항목 | 유사도 검색 (similarity) | MMR 검색 |
|------|------------------------|----------|
| 기준 | 관련성만 고려 | 관련성 + 다양성 동시 고려 |
| 장점 | 가장 관련 높은 문서 보장 | 다양한 관점의 문서 수집 |
| 단점 | 중복 청크 가능 | 관련성이 약간 낮을 수 있음 |
| `lambda_mult` | 해당 없음 | 0=최대 다양성, 1=최대 관련성 |

> 💡 **실무 팁**: 긴 문서에서 같은 섹션의 청크가 반복적으로 선택될 때 MMR이 효과적이에요. `lambda_mult` 파라미터로 관련성(1.0)과 다양성(0.0) 사이의 균형을 조절해요. 일반적으로 `0.5`가 좋은 출발점이에요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **2-Step RAG**: 검색 → 생성의 선형 파이프라인. 단순하고 빠르며 디버깅이 쉬워요. `TypedDict` 상태에 `question`, `documents`, `generation`을 정의해요.
- **FAISS + as_retriever()**: `search_type`(similarity/mmr)과 `search_kwargs`(k, fetch_k, filter)로 검색 동작을 세밀하게 조정해요.
- **Agentic RAG**: `create_agent` + `@tool` 데코레이터로 에이전트가 검색 도구 사용 여부를 자율 결정해요. docstring이 도구 선택의 핵심이에요.
- **Hybrid RAG**: 쿼리 강화(Query Enhancement)로 N개의 다양한 쿼리를 생성하고, 중복 제거 후 더 많은 관련 문서를 수집해요. 검증 루프로 품질을 보장하되 `max_retries`로 무한 루프를 방지해요.
- **메타데이터 필터링**: `filter` 파라미터로 특정 페이지/소스 범위에서만 검색을 제한해요. 다중 문서 시스템에서 필수적인 기법이에요.
- **MMR 검색**: `search_type="mmr"`로 관련성과 다양성을 동시에 고려하여 중복된 청크 선택을 줄여요.

### Part 08 RAG 전체 정리

Part 08에서 배운 모든 RAG 기법을 한눈에 정리해요:

| 노트북 | 핵심 기법 | 해결하는 문제 | 비용 수준 |
|--------|----------|-------------|----------|
| **01** Naive RAG | Retrieve → Generate | 기본 문서 Q&A | 낮음 |
| **02** Agentic RAG | 관련성 체크 + 웹 폴백 + 에이전트 자율 판단 | 무한 루프, 불필요한 검색 | 중간 |
| **03** CRAG/Self-RAG | 검색 보강(CRAG) + 생성 품질 검증(Self-RAG) | 검색 실패, 환각 | 중-높음 |
| **04** Adaptive RAG | 사전 라우팅 + 3단계 검증 | 소스 선택, 비용 최적화 | 높음 |
| **05** Retrieval | 쿼리 강화 + MMR + 메타데이터 필터링 | 검색 품질 자체 개선 | 가변 |

> 🔗 **벡터 RAG를 넘어서 — 관계형 질문이 주가 되는 경우**: 위 5가지는 모두 **벡터 검색**을 중심으로 해요. 데이터가 문서가 아니라 **엔티티·관계의 그래프**라면(예: "이 약과 상호작용하는 약은?", "이 인물과 공저자인 사람은?"), `11_Use_Cases/06-GraphRAG-Neo4j.ipynb`의 **Text2Cypher** 파이프라인이 더 정확해요. 벡터 RAG와 GraphRAG는 대체재가 아니라 **보완재**이며, 실제 프로덕션에서는 Part 09의 Router·Supervisor로 둘을 병렬 배치하는 경우가 많아요.

> 💡 **핵심 정리**: RAG 시스템 구축은 **점진적 개선**이 핵심이에요. Naive RAG로 시작해서 LangSmith로 병목을 찾고, 필요한 기법만 추가하세요. "처음부터 Adaptive RAG"는 과잉 엔지니어링이에요.

### 5가지 사고 도구 관점에서 본 Part 08

| 패턴 | 주로 사용한 도구 | 역할 |
|------|----------------|------|
| **Naive RAG** (01) | **Inform** | 검색 결과를 프롬프트에 주입 |
| **Agentic RAG** (02) | Inform + **Verify** + **Constrain** | 관련성 그레이더 + 에이전트가 검색 필요성 자율 판단 |
| **CRAG / Self-RAG** (03) | Inform + Verify + **Correct** | 관련성·환각 점검 후 쿼리 재작성 또는 웹 폴백으로 교정 |
| **Adaptive RAG** (04) | 네 도구(Inform+Constrain+Verify+Correct) 전부 | 라우팅으로 Constrain, 3단계 체크로 Verify/Correct |
| **Retrieval 심화** (05) | Inform + Constrain | 메타데이터 필터·MMR로 컨텍스트 품질과 범위 제한 |

## 다음 노트북 예고 — Part 09 Multi-Agent로 이어져요

이번 챕터에서 완성한 RAG 패턴들은 **혼자 일하는 에이전트가 검색을 잘 쓰는 법**이에요. Part 09에서는 시야를 넓혀, 여러 에이전트가 역할을 나눠 협력하는 **멀티에이전트 아키텍처**를 배워요.

- **`09_Multi_Agent/01-Multi-Agent-Overview.ipynb`**: Supervisor / Collaboration / Handoff / Router / Hierarchical 다섯 가지 협력 패턴 개관
- RAG와의 연결: 여기서 만든 Adaptive RAG 그래프는 **Researcher 에이전트의 sub-graph**로 그대로 끼워 넣을 수 있어요. 예를 들어 Supervisor 밑에 "문서 RAG 에이전트 + 웹 검색 에이전트 + 코드 분석 에이전트"를 두면, 이번 챕터의 Adaptive RAG가 그중 한 Worker가 돼요.
- 이후 **Part 10 Deep Agents**에서는 이 멀티에이전트 구조를 한 단계 더 추상화한 프레임워크(파일시스템·서브에이전트·장기 메모리 내장)를 만나요.

> 💡 **학습 전략**: 멀티에이전트 패턴을 공부할 때도 **5가지 사고 도구**로 각 패턴을 분해해보세요. Supervisor = Constrain(경로 제한) + Inform(상태 공유), Evaluator subgraph = Verify + Correct — 이런 식으로 이름이 바뀌어도 근본 원리는 그대로예요.